In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files into memory (store full dataframes for later steps)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error_logs = pd.read_csv('error_logs.csv')

# Helper: convert timestamp column to timezone-aware datetime (Asia/Shanghai) and return min/max as strings
def ts_min_max(df, ts_col='timestamp'):
    # Always return ((min_epoch, min_str), (max_epoch, max_str)) with None placeholders if unavailable
    if ts_col not in df.columns or df[ts_col].isnull().all():
        return (None, None), (None, None)
    try:
        s = pd.to_datetime(df[ts_col].dropna().astype(int), unit='s', utc=True)
        if s.empty:
            return (None, None), (None, None)
        s = s.dt.tz_convert(tz)
        mn = s.min()
        mx = s.max()
        return (int(mn.timestamp()), mn.strftime('%Y-%m-%d %H:%M:%S %Z')), (int(mx.timestamp()), mx.strftime('%Y-%m-%d %H:%M:%S %Z'))
    except Exception:
        return (None, None), (None, None)

# Helper: top N unique sorted
def unique_sorted_limited(series, n=20):
    if series is None:
        return []
    uniques = sorted(series.dropna().astype(str).unique())
    return uniques[:n]

# Summaries for metric.csv
metric_rows = df_metric.shape[0]
(metric_min_epoch, metric_min_str), (metric_max_epoch, metric_max_str) = ts_min_max(df_metric, 'timestamp')
metric_cmdbs = unique_sorted_limited(df_metric.get('cmdb_id', pd.Series(dtype=str)), 20)
metric_kpis = unique_sorted_limited(df_metric.get('kpi_name', pd.Series(dtype=str)), 20)

summary_metric = pd.DataFrame([{
    'file': 'metric.csv',
    'rows': metric_rows,
    'min_timestamp_epoch': metric_min_epoch,
    'min_timestamp_local': metric_min_str,
    'max_timestamp_epoch': metric_max_epoch,
    'max_timestamp_local': metric_max_str,
    'unique_cmdb_ids_shown': metric_cmdbs,
    'unique_kpi_names_shown': metric_kpis
}])

# Compute global P50 and P95 per (cmdb_id, kpi_name) pair for metric.csv
if {'value','cmdb_id','kpi_name'}.issubset(df_metric.columns):
    grp_metric = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    ).reset_index().sort_values('count', ascending=False).head(20)
    top_metric_pairs = grp_metric.rename(columns={'kpi_name': 'KPI_NAME'})[['cmdb_id', 'KPI_NAME', 'count', 'p50', 'p95']].reset_index(drop=True)
    top_metric_pairs['p50'] = top_metric_pairs['p50'].round(6)
    top_metric_pairs['p95'] = top_metric_pairs['p95'].round(6)
else:
    top_metric_pairs = pd.DataFrame(columns=['cmdb_id','KPI_NAME','count','p50','p95'])

# Summaries for trace.csv
trace_rows = df_trace.shape[0]
(trace_min_epoch, trace_min_str), (trace_max_epoch, trace_max_str) = ts_min_max(df_trace, 'timestamp')
trace_cmdbs = unique_sorted_limited(df_trace.get('cmdb_id', pd.Series(dtype=str)), 20)
trace_names = unique_sorted_limited(df_trace.get('trace_name', pd.Series(dtype=str)), 20)

summary_trace = pd.DataFrame([{
    'file': 'trace.csv',
    'rows': trace_rows,
    'min_timestamp_epoch': trace_min_epoch,
    'min_timestamp_local': trace_min_str,
    'max_timestamp_epoch': trace_max_epoch,
    'max_timestamp_local': trace_max_str,
    'unique_cmdb_ids_shown': trace_cmdbs,
    'unique_trace_names_shown': trace_names
}])

# Compute global P50 and P95 per (cmdb_id, trace_name) pair for trace.csv
if {'value','cmdb_id','trace_name'}.issubset(df_trace.columns):
    grp_trace = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        count='count',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    ).reset_index().sort_values('count', ascending=False).head(20)
    top_trace_pairs = grp_trace.rename(columns={'trace_name': 'TRACE_NAME'})[['cmdb_id', 'TRACE_NAME', 'count', 'p50', 'p95']].reset_index(drop=True)
    top_trace_pairs['p50'] = top_trace_pairs['p50'].round(6)
    top_trace_pairs['p95'] = top_trace_pairs['p95'].round(6)
else:
    top_trace_pairs = pd.DataFrame(columns=['cmdb_id','TRACE_NAME','count','p50','p95'])

# Summaries for log.csv
log_rows = df_log.shape[0]
(log_min_epoch, log_min_str), (log_max_epoch, log_max_str) = ts_min_max(df_log, 'timestamp')
log_cmdbs = unique_sorted_limited(df_log.get('cmdb_id', pd.Series(dtype=str)), 20)
log_names = unique_sorted_limited(df_log.get('log_name', pd.Series(dtype=str)), 20)

summary_log = pd.DataFrame([{
    'file': 'log.csv',
    'rows': log_rows,
    'min_timestamp_epoch': log_min_epoch,
    'min_timestamp_local': log_min_str,
    'max_timestamp_epoch': log_max_epoch,
    'max_timestamp_local': log_max_str,
    'unique_cmdb_ids_shown': log_cmdbs,
    'unique_log_names_shown': log_names
}])

# Summaries for error_logs.csv
err_rows = df_error_logs.shape[0]
(err_min_epoch, err_min_str), (err_max_epoch, err_max_str) = ts_min_max(df_error_logs, 'timestamp')
err_cmdbs = unique_sorted_limited(df_error_logs.get('cmdb_id', pd.Series(dtype=str)), 20)
err_messages = unique_sorted_limited(df_error_logs.get('message', pd.Series(dtype=str)), 20)

summary_error = pd.DataFrame([{
    'file': 'error_logs.csv',
    'rows': err_rows,
    'min_timestamp_epoch': err_min_epoch,
    'min_timestamp_local': err_min_str,
    'max_timestamp_epoch': err_max_epoch,
    'max_timestamp_local': err_max_str,
    'unique_cmdb_ids_shown': err_cmdbs,
    'unique_messages_shown': err_messages
}])

# Final compact outputs (full dataframes remain in df_metric, df_trace, df_log, df_error_logs)
summary_metric, top_metric_pairs, summary_trace, top_trace_pairs, summary_log, summary_error

```
Out[1]:
```
metric_rows, metric_min_str, metric_max_str, metric_cmdbs_shown, metric_kpis_shown = metric_rows, metric_min_str, metric_max_str, metric_cmdbs, metric_kpis
trace_rows, trace_min_str, trace_max_str, trace_cmdbs_shown, trace_names_shown = trace_rows, trace_min_str, trace_max_str, trace_cmdbs, trace_names
log_rows, log_min_str, log_max_str, log_cmdbs_shown, log_names_shown = log_rows, log_min_str, log_max_str, log_cmdbs, log_names
err_rows, err_min_str, err_max_str, err_cmdbs_shown = err_rows, err_min_str, err_max_str, err_cmdbs

metric_unique_cmdb_count = df_metric['cmdb_id'].nunique() if 'cmdb_id' in df_metric.columns else 0
metric_unique_kpi_count = df_metric['kpi_name'].nunique() if 'kpi_name' in df_metric.columns else 0
trace_unique_cmdb_count = df_trace['cmdb_id'].nunique() if 'cmdb_id' in df_trace.columns else 0
trace_unique_name_count = df_trace['trace_name'].nunique() if 'trace_name' in df_trace.columns else 0
log_unique_cmdb_count = df_log['cmdb_id'].nunique() if 'cmdb_id' in df_log.columns else 0
log_unique_name_count = df_log['log_name'].nunique() if 'log_name' in df_log.columns else 0
err_unique_cmdb_count = df_error_logs['cmdb_id'].nunique() if 'cmdb_id' in df_error_logs.columns else 0

# Prepare compact lists (already limited to up to 20 in earlier step)
metric_cmdb_list = metric_cmdbs_shown
metric_kpi_list = metric_kpis_shown
trace_cmdb_list = trace_cmdbs_shown
trace_name_list = trace_names_shown
log_cmdb_list = log_cmdbs_shown
log_name_list = log_names_shown
err_cmdb_list = err_cmdbs_shown

# Top metric/trace pairs summaries (already computed as DataFrames)
top_metric_pairs_small = top_metric_pairs[['cmdb_id','KPI_NAME','count','p50','p95']].head(5) if not top_metric_pairs.empty else None
top_trace_pairs_small = top_trace_pairs[['cmdb_id','TRACE_NAME','count','p50','p95']].head(5) if not top_trace_pairs.empty else None

summary_text = (
    "Telemetry files summary:\n\n"
    "1) metric.csv:\n"
    f" - Total rows: {metric_rows}\n"
    f" - Time range (Asia/Shanghai): {metric_min_str} to {metric_max_str}\n"
    f" - Unique cmdb_id count: {metric_unique_cmdb_count}; sample (up to 20, sorted): {metric_cmdb_list}\n"
    f" - Unique KPI count: {metric_unique_kpi_count}; sample (up to 20): {metric_kpi_list}\n"
    " - Top (cmdb_id, KPI) pairs by row count (top 5 shown):\n"
    f"   {top_metric_pairs_small.to_dict(orient='records') if top_metric_pairs_small is not None else 'None'}\n\n"
    "2) trace.csv:\n"
    f" - Total rows: {trace_rows}\n"
    f" - Time range (Asia/Shanghai): {trace_min_str} to {trace_max_str}\n"
    f" - Unique cmdb_id count: {trace_unique_cmdb_count}; sample (up to 20, sorted): {trace_cmdb_list}\n"
    f" - Unique trace name count: {trace_unique_name_count}; sample (up to 20): {trace_name_list}\n"
    " - Top (cmdb_id, TRACE_NAME) pairs by row count (top 5 shown):\n"
    f"   {top_trace_pairs_small.to_dict(orient='records') if top_trace_pairs_small is not None else 'None'}\n\n"
    "3) log.csv:\n"
    f" - Total rows: {log_rows}\n"
    f" - Time range (Asia/Shanghai): {log_min_str} to {log_max_str}\n"
    f" - Unique cmdb_id count: {log_unique_cmdb_count}; sample (up to 20, sorted): {log_cmdb_list}\n"
    f" - Unique log name count: {log_unique_name_count}; sample (up to 20): {log_name_list}\n\n"
    "4) error_logs.csv:\n"
    f" - Total rows: {err_rows}\n"
    f" - Time range: {err_min_str} to {err_max_str}\n"
    f" - Unique cmdb_id count: {err_unique_cmdb_count}; sample (up to 20): {err_cmdb_list}\n\n"
    "Notes: error_logs.csv is empty. Metric data covers ~1 hour (15:30–16:30 CST) with 319,703 rows; "
    "trace data covers ~29 minutes (16:00–16:29 CST) with 35,976 rows. Top pairs lists include count, P50 and P95."
)

summary_text[:10000]  # display the summary (kept compact)

The original code execution output of IPython Kernel is also provided below for reference:

(         file    rows  min_timestamp_epoch      min_timestamp_local  max_timestamp_epoch      max_timestamp_local                              unique_cmdb_ids_shown                             unique_kpi_names_shown
0  metric.csv  319703           1647761400  2022-03-20 15:30:00 CST           1647765000  2022-03-20 16:30:00 CST  [adservice, adservice-0, adservice-1, adservic...  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,                cmdb_id                                           KPI_NAME  count           p50           p95
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  1.542250e+03  1.890500e+03
1            adservice                                     app.grpc.count     61  2.370000e+02  2.610000e+02
2            adservice                                       app.grpc.mrt     61  2.497000e-03  2.599000e-03
3            adservice                                        app.grpc.rr     61  1.000000e+02  1.000000e+02
4            adservice                                        app.grpc.sr     61  1.000000e+02  1.000000e+02
5   shippingservice2-0  container.node-6.container_network_receive_MB....     61  3.631100e-02  4.420800e-02
6   shippingservice2-0  container.node-6.container_network_receive_err...     61  0.000000e+00  0.000000e+00
7   shippingservice2-0  container.node-6.container_network_receive_pac...     61  3.525000e+02  4.125000e+02
8   shippingservice2-0  container.node-6.container_network_receive_pac...     61  0.000000e+00  0.000000e+00
9   shippingservice2-0  container.node-6.container_network_transmit_MB...     61  4.744820e-01  4.948870e-01
10  shippingservice2-0  container.node-6.container_network_transmit_er...     61  0.000000e+00  0.000000e+00
11  shippingservice2-0  container.node-6.container_network_transmit_pa...     61  2.300000e+02  2.700000e+02
12  shippingservice2-0  container.node-6.container_network_transmit_pa...     61  0.000000e+00  0.000000e+00
13  shippingservice2-0  container.node-6.container_memory_failures.hie...     61  0.000000e+00  3.300000e+01
14  shippingservice2-0  container.node-6.container_memory_failures.hie...     61  0.000000e+00  0.000000e+00
15  shippingservice2-0      container.node-6.container_memory_mapped_file     61  2.703360e+05  2.703360e+05
16  shippingservice2-0     container.node-6.container_memory_max_usage_MB     61  1.287266e+02  1.287266e+02
17  shippingservice2-0              container.node-6.container_memory_rss     61  8.871936e+06  8.871936e+06
18  shippingservice2-0             container.node-6.container_memory_swap     61  0.000000e+00  0.000000e+00
19  shippingservice2-0         container.node-6.container_memory_usage_MB     61  4.265234e+01  4.283203e+01,         file   rows  min_timestamp_epoch      min_timestamp_local  max_timestamp_epoch      max_timestamp_local                              unique_cmdb_ids_shown                           unique_trace_names_shown
0  trace.csv  35976           1647763200  2022-03-20 16:00:00 CST           1647764940  2022-03-20 16:29:00 CST  [adservice-0, adservice-1, adservice-2, adserv...  [trace.from_checkoutservice-0.duration_mean, t...,                cmdb_id                           TRACE_NAME  count        p50        p95
0   shippingservice2-0     trace.from_frontend2-0.row_count     30  15.000000  25.100000
1          adservice-0  trace.from_frontend-0.duration_mean     30   0.000018   0.000022
2          adservice-0   trace.from_frontend-0.duration_p95     30   0.000026   0.000033
3          adservice-0     trace.from_frontend-0.error_rate     30   0.000000   0.000000
4          adservice-0      trace.from_frontend-0.row_count     30  31.000000  34.550000
5          adservice-0  trace.from_frontend-1.duration_mean     30   0.000018   0.000026
6          adservice-0   trace.from_frontend-1.duration_p95     30   0.000023   0.000036
7          adservice-0     trace.from_frontend-1.error_rate     30   0.000000   0.000000
8          adservice-0      trace.from_frontend-1.row_count     30   3.000000   7.000000
9          adservice-0  trace.from_frontend-2.duration_mean     30   0.000019   0.000021
10         adservice-0   trace.from_frontend-2.duration_p95     30   0.000027   0.000033
11         adservice-0     trace.from_frontend-2.error_rate     30   0.000000   0.000000
12         adservice-0      trace.from_frontend-2.row_count     30  31.500000  33.550000
13         adservice-1  trace.from_frontend-0.duration_mean     30   0.000017   0.000022
14         adservice-1   trace.from_frontend-0.duration_p95     30   0.000022   0.000037
15         adservice-1     trace.from_frontend-0.error_rate     30   0.000000   0.000000
16   shippingservice-1  trace.from_frontend-2.duration_mean     30   0.000071   0.000077
17   shippingservice-1   trace.from_frontend-2.duration_p95     30   0.000080   0.000096
18   shippingservice-1     trace.from_frontend-2.error_rate     30   0.000000   0.000000
19   shippingservice-1      trace.from_frontend-2.row_count     30  11.000000  12.550000,       file  rows  min_timestamp_epoch      min_timestamp_local  max_timestamp_epoch      max_timestamp_local                              unique_cmdb_ids_shown            unique_log_names_shown
0  log.csv  1840           1647763200  2022-03-20 16:00:00 CST           1647765000  2022-03-20 16:30:00 CST  [adservice-0, adservice-1, adservice-2, cartse...  [log.error_count, log.row_count],              file  rows min_timestamp_epoch min_timestamp_local max_timestamp_epoch max_timestamp_local unique_cmdb_ids_shown unique_messages_shown
0  error_logs.csv     0                None                None                None                None                    []                    [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric already in kernel
# 1) Compute global P50 and P95 per (cmdb_id, kpi_name) using the entire metric series
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95))
    .reset_index()
)
metric_thresholds['p50'] = metric_thresholds['p50'].round(6)
metric_thresholds['p95'] = metric_thresholds['p95'].round(6)

# 2) Filter metric rows to the incident window (inclusive)
start_ts = 1647763200
end_ts = 1647765000
metric_window_df = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Annotate window rows with global P95 and detect anomalies (value > global P95)
metric_window_df = metric_window_df.merge(
    metric_thresholds[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)
metric_window_df['is_anom'] = metric_window_df['value'] > metric_window_df['p95']

# Aggregations within the window
total_counts = metric_window_df.groupby(['cmdb_id', 'kpi_name']).size().reset_index(name='total_count')
anomaly_counts = metric_window_df.groupby(['cmdb_id', 'kpi_name'])['is_anom'].sum().reset_index(name='anomaly_count')
max_values = metric_window_df.groupby(['cmdb_id', 'kpi_name'])['value'].max().reset_index(name='max_value')
earliest_anoms = (
    metric_window_df[metric_window_df['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .reset_index(name='earliest_anom_timestamp')
)

# Merge aggregated results
agg = total_counts.merge(anomaly_counts, on=['cmdb_id', 'kpi_name'], how='left')
agg = agg.merge(max_values, on=['cmdb_id', 'kpi_name'], how='left')
agg = agg.merge(earliest_anoms, on=['cmdb_id', 'kpi_name'], how='left')
agg = agg.merge(metric_thresholds[['cmdb_id', 'kpi_name', 'p50', 'p95']], on=['cmdb_id', 'kpi_name'], how='left')

# Convert earliest_anom_timestamp to Asia/Shanghai localized string, keep NULL as pd.NA
def to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

agg['earliest_anom_timestamp'] = agg['earliest_anom_timestamp'].apply(lambda x: to_local_str(x))

# Prepare final result: top 20 by anomaly_count desc
result_df = agg.sort_values(['anomaly_count', 'total_count'], ascending=[False, False]).reset_index(drop=True)
result_df_top20 = result_df[['cmdb_id', 'kpi_name', 'total_count', 'anomaly_count', 'earliest_anom_timestamp', 'max_value', 'p50', 'p95']].head(20)

# Total number of (cmdb_id, kpi_name) pairs that had anomaly_count > 0
anomaly_pairs_count = int((result_df['anomaly_count'] > 0).sum())

# Keep intermediate tables in kernel as required: metric_thresholds, metric_window_df
metric_thresholds, metric_window_df, result_df_top20, anomaly_pairs_count

```
Out[2]:
```
summary = (
    "Summary of metric anomaly analysis (plain English):\n\n"
    "- Global thresholds (P50, P95) were computed and saved in variable 'metric_thresholds' for 5,253 (cmdb_id, kpi_name) pairs.\n"
    "- The incident window (2022-03-20 16:00:00 to 16:30:00 CST; epoch 1647763200–1647765000) contains 162,473 metric rows stored in 'metric_window_df'.\n"
    "- Per-pair aggregations were computed for the window. The top 20 pairs by anomaly_count are available in 'result_df_top20'.\n"
    "- Total number of (cmdb_id, kpi_name) pairs with anomaly_count > 0: 1,525.\n"
    "- Observed pattern in the top results: many pairs (e.g., JVM/runtime and some container fs metrics) have anomaly_count equal to their total_count (31), with earliest anomaly at the start of the window (2022-03-20 16:00:00 CST), meaning their values were above the global P95 throughout the window.\n\n"
    "You can inspect the top 20 table in 'result_df_top20', and the full thresholds in 'metric_thresholds', for details."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count           p50           p95
0              adservice                                     app.grpc.count     61    237.000000    261.000000
1              adservice                                       app.grpc.mrt     61      0.002497      0.002599
2              adservice                                        app.grpc.rr     61    100.000000    100.000000
3              adservice                                        app.grpc.sr     61    100.000000    100.000000
4              adservice                                     app.http.count     61     24.000000     24.000000
...                  ...                                                ...    ...           ...           ...
5248  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     61     15.500000     21.000000
5249  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  47025.000000  58000.000000
5250  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61      0.000525      0.002575
5251  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61     15.500000     19.000000
5252  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61   1542.250000   1890.500000

[5253 rows x 5 columns],          timestamp             cmdb_id                                           kpi_name         value           p95  is_anom
0       1647763200           adservice                                     app.grpc.count    233.000000    261.000000    False
1       1647763200           adservice                                       app.grpc.mrt      0.002522      0.002599    False
2       1647763200           adservice                                        app.grpc.rr    100.000000    100.000000    False
3       1647763200           adservice                                        app.grpc.sr    100.000000    100.000000    False
4       1647763200           adservice                                     app.http.count     24.000000     24.000000    False
...            ...                 ...                                                ...           ...           ...      ...
162468  1647765000  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      8.500000     21.000000    False
162469  1647765000  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  28375.000000  58000.000000    False
162470  1647765000  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.000525      0.002575    False
162471  1647765000  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      8.500000     19.000000    False
162472  1647765000  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    845.750000   1890.500000    False

[162473 rows x 6 columns],         cmdb_id                                           kpi_name  total_count  anomaly_count  earliest_anom_timestamp     max_value           p50           p95
0     adservice                runtime.java_lang_Runtime_StartTime           31             31  2022-03-20 16:00:00 CST  1.647144e+12  1.647144e+12  1.647144e+12
1     adservice               runtime.jvm_memory_MB_committed.heap           31             31  2022-03-20 16:00:00 CST  9.532031e+01  9.532031e+01  9.532031e+01
2     adservice                  runtime.jvm_memory_MB_max.nonheap           31             31  2022-03-20 16:00:00 CST -9.536743e-07 -1.000000e-06 -1.000000e-06
3     adservice                 runtime.jvm_memory_MB_used.nonheap           31             31  2022-03-20 16:00:00 CST  5.521247e+01  5.521247e+01  5.521247e+01
4     adservice   runtime.jvm_memory_pool_MB_committed.Tenured_Gen           31             31  2022-03-20 16:00:00 CST  6.569531e+01  6.569531e+01  6.569531e+01
5     adservice           runtime.jvm_memory_pool_MB_max.Metaspace           31             31  2022-03-20 16:00:00 CST -9.536743e-07 -1.000000e-06 -1.000000e-06
6     adservice          runtime.jvm_memory_pool_MB_used.Metaspace           31             31  2022-03-20 16:00:00 CST  3.002809e+01  3.002809e+01  3.002809e+01
7     adservice  runtime.jvm_memory_pool_allocated_MB_created.C...           31             31  2022-03-20 16:00:00 CST  1.570839e+03  1.570839e+03  1.570839e+03
8     adservice  runtime.jvm_memory_pool_allocated_MB_created.C...           31             31  2022-03-20 16:00:00 CST  1.570839e+03  1.570839e+03  1.570839e+03
9     adservice  runtime.jvm_memory_pool_allocated_MB_created.E...           31             31  2022-03-20 16:00:00 CST  1.570839e+03  1.570839e+03  1.570839e+03
10    adservice  runtime.jvm_memory_pool_allocated_MB_created.M...           31             31  2022-03-20 16:00:00 CST  1.570839e+03  1.570839e+03  1.570839e+03
11    adservice  runtime.jvm_memory_pool_allocated_MB_created.S...           31             31  2022-03-20 16:00:00 CST  1.570839e+03  1.570839e+03  1.570839e+03
12    adservice  runtime.jvm_memory_pool_allocated_MB_created.T...           31             31  2022-03-20 16:00:00 CST  1.570839e+03  1.570839e+03  1.570839e+03
13    adservice  runtime.jvm_memory_pool_allocated_MB_total.Com...           31             31  2022-03-20 16:00:00 CST  3.473007e+00  3.473007e+00  3.473007e+00
14    adservice  runtime.jvm_memory_pool_allocated_MB_total.Met...           31             31  2022-03-20 16:00:00 CST  3.009750e+01  3.009750e+01  3.009750e+01
15  adservice-0   container.node-6.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05  6.046307e+05
16  adservice-0   container.node-6.container_fs_usage_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  2.638281e+01  2.638281e+01  2.638281e+01
17  adservice-1   container.node-6.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05  6.046307e+05
18  adservice-2   container.node-5.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05  6.046307e+05
19   adservice2                  runtime.jvm_memory_MB_max.nonheap           31             31  2022-03-20 16:00:00 CST -9.536743e-07 -1.000000e-06 -1.000000e-06, 1525)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing per-(cmdb_id,kpi_name) aggregation DataFrame 'result_df' (created earlier).
# If not present, fall back to 'agg' which was also used earlier.
per_kpi = result_df if 'result_df' in globals() else agg

# Ensure required columns exist
required_cols = {'cmdb_id', 'kpi_name', 'anomaly_count', 'max_value', 'p95'}
missing = required_cols - set(per_kpi.columns)
if missing:
    raise KeyError(f"Missing required columns in per-kpi DataFrame: {missing}")

# 1) Compute breach ratio per KPI: max_value / p95 if p95>0 else use max_value
per_kpi = per_kpi.copy()
per_kpi['p95'] = per_kpi['p95'].astype(float)
per_kpi['max_value'] = per_kpi['max_value'].astype(float)
per_kpi['breach_ratio'] = np.where(per_kpi['p95'] > 0, per_kpi['max_value'] / per_kpi['p95'], per_kpi['max_value'])
per_kpi['breach_ratio'] = per_kpi['breach_ratio'].replace([np.inf, -np.inf], np.nan)

# 2) Aggregate per-component for KPIs that had anomaly_count > 0
faulty_kpis = per_kpi[per_kpi['anomaly_count'] > 0].copy()

component_agg = (
    faulty_kpis
    .groupby('cmdb_id')
    .agg(
        num_faulty_kpis=('kpi_name', 'nunique'),
        total_anomaly_points=('anomaly_count', 'sum'),
        worst_breach_ratio=('breach_ratio', 'max'),
        max_value_overall=('max_value', 'max')
    )
    .reset_index()
)

# 3) Compute earliest_anomaly per component from metric_window_df (use epoch min where is_anom is True)
if 'metric_window_df' in globals():
    earliest_per_comp = (
        metric_window_df[metric_window_df['is_anom']]
        .groupby('cmdb_id')['timestamp']
        .min()
        .reset_index(name='earliest_anom_epoch')
    )
    # convert epoch to Asia/Shanghai localized string
    def epoch_to_local_str(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    earliest_per_comp['earliest_anomaly'] = earliest_per_comp['earliest_anom_epoch'].apply(epoch_to_local_str)
    earliest_per_comp = earliest_per_comp[['cmdb_id', 'earliest_anomaly']]
else:
    # If metric_window_df not present, set earliest as NA
    earliest_per_comp = pd.DataFrame(columns=['cmdb_id', 'earliest_anomaly'])

# 4) Merge earliest info into component_agg
component_anom_summary = component_agg.merge(earliest_per_comp, on='cmdb_id', how='left')

# Reorder columns and round numeric severity for compactness
component_anom_summary = component_anom_summary[
    ['cmdb_id', 'num_faulty_kpis', 'total_anomaly_points', 'earliest_anomaly', 'worst_breach_ratio', 'max_value_overall']
]
component_anom_summary['worst_breach_ratio'] = component_anom_summary['worst_breach_ratio'].round(6)
component_anom_summary['max_value_overall'] = component_anom_summary['max_value_overall'].round(6)

# Sort by num_faulty_kpis desc, then total_anomaly_points desc
component_anom_summary = component_anom_summary.sort_values(['num_faulty_kpis', 'total_anomaly_points'], ascending=[False, False]).reset_index(drop=True)

# Total number of components with num_faulty_kpis > 0
component_count = int((component_anom_summary['num_faulty_kpis'] > 0).sum())

# Top 10 cmdb_id values by the same rank
top10_cmdb = component_anom_summary['cmdb_id'].head(10).reset_index(drop=True)

# Keep output compact: store summary in kernel variable and display up to 20 rows
component_anom_summary.head(20), component_count, top10_cmdb

```
Out[3]:
```
python
summary = (
    "Compact summary of per-component anomalies (plain English):\n\n"
    "- Total components with at least one faulty KPI: 58.\n\n"
    "- Top 5 affected components (num_faulty_kpis, total_anomaly_points, worst_breach_ratio, earliest anomaly):\n"
    "  1) frontend-1: 65 faulty KPIs, 218 anomaly points, worst breach ratio 3.00, earliest 2022-03-20 16:00:00 CST\n"
    "  2) checkoutservice-1: 60 faulty KPIs, 201 anomaly points, worst breach ratio 33.00, earliest 2022-03-20 16:00:00 CST\n"
    "  3) frontend-0: 60 faulty KPIs, 138 anomaly points, worst breach ratio 33.00, earliest 2022-03-20 16:00:00 CST\n"
    "  4) frontend-2: 58 faulty KPIs, 134 anomaly points, worst breach ratio 6.00, earliest 2022-03-20 16:00:00 CST\n"
    "  5) checkoutservice-2: 56 faulty KPIs, 100 anomaly points, worst breach ratio ~4.63, earliest 2022-03-20 16:00:00 CST\n\n"
    "- Top 10 components by this rank (in order): frontend-1, checkoutservice-1, frontend-0, frontend-2, "
    "checkoutservice-2, frontend2-0, checkoutservice-0, productcatalogservice-1, adservice, productcatalogservice-2.\n\n"
    "Notes: Many components show earliest anomalies at the start of the incident window (2022-03-20 16:00:00 CST). "
    "Worst_breach_ratio is max_value / global P95 (or max_value when P95==0) and highlights how far values exceeded normal."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  num_faulty_kpis  total_anomaly_points         earliest_anomaly  worst_breach_ratio  max_value_overall
0                frontend-1               65                   218  2022-03-20 16:00:00 CST            3.000000       1.647765e+09
1         checkoutservice-1               60                   201  2022-03-20 16:00:00 CST           33.000000       1.647765e+09
2                frontend-0               60                   138  2022-03-20 16:00:00 CST           33.000000       1.647765e+09
3                frontend-2               58                   134  2022-03-20 16:00:00 CST            6.000000       1.647765e+09
4         checkoutservice-2               56                   100  2022-03-20 16:00:00 CST            4.631115       1.647765e+09
5               frontend2-0               50                   139  2022-03-20 16:00:00 CST            2.624558       1.647765e+09
6         checkoutservice-0               49                   121  2022-03-20 16:00:00 CST           22.539683       1.647765e+09
7   productcatalogservice-1               46                   194  2022-03-20 16:00:00 CST            1.581699       1.647765e+09
8                 adservice               45                   543  2022-03-20 16:00:00 CST            2.250000       1.995013e+12
9   productcatalogservice-2               45                   171  2022-03-20 16:00:00 CST           49.500000       1.647765e+09
10  productcatalogservice-0               41                   154  2022-03-20 16:00:00 CST           33.000000       1.647765e+09
11           cartservice2-0               39                    95  2022-03-20 16:00:00 CST         1875.000000       1.647765e+09
12               adservice2               38                   231  2022-03-20 16:00:00 CST            1.661969       1.608288e+12
13                   node-6               35                    82  2022-03-20 16:00:00 CST           24.500000       6.596809e+09
14            cartservice-2               34                   110  2022-03-20 16:00:00 CST            6.681897       1.647765e+09
15            cartservice-0               32                   111  2022-03-20 16:00:00 CST            6.726879       1.647765e+09
16        shippingservice-2               29                    87  2022-03-20 16:00:00 CST           11.967652       1.647765e+09
17        shippingservice-0               29                    86  2022-03-20 16:00:00 CST           12.901907       1.647765e+09
18                   node-3               29                    56  2022-03-20 16:00:00 CST            3.400000       1.089937e+10
19  recommendationservice-1               28                   134  2022-03-20 16:00:00 CST            5.000000       1.647765e+09, 58, 0                 frontend-1
1          checkoutservice-1
2                 frontend-0
3                 frontend-2
4          checkoutservice-2
5                frontend2-0
6          checkoutservice-0
7    productcatalogservice-1
8                  adservice
9    productcatalogservice-2
Name: cmdb_id, dtype: object)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing per-KPI aggregated DataFrame (prefer 'result_df', else fallback to 'agg')
per_kpi = globals().get('result_df', globals().get('agg', None))
if per_kpi is None:
    raise RuntimeError("Per-KPI aggregation DataFrame ('result_df' or 'agg') not found in kernel.")

# Ensure expected columns exist; provide fallbacks for earliest timestamp column name
if 'earliest_anom_timestamp' not in per_kpi.columns and 'earliest_anomaly' in per_kpi.columns:
    per_kpi = per_kpi.rename(columns={'earliest_anomaly': 'earliest_anom_timestamp'})

# Required columns check
required = {'cmdb_id', 'kpi_name', 'total_count', 'anomaly_count', 'max_value', 'p95', 'earliest_anom_timestamp'}
missing = required - set(per_kpi.columns)
if missing:
    # Allow earliest column to be missing by filling with NA
    if 'earliest_anom_timestamp' in missing:
        per_kpi['earliest_anom_timestamp'] = pd.NA
        missing.remove('earliest_anom_timestamp')
    if missing:
        raise KeyError(f"Missing required columns in per-kpi DataFrame: {missing}")

# 1) Top 5 components by given rank
top_components = ['frontend-1', 'checkoutservice-1', 'frontend-0', 'frontend-2', 'checkoutservice-2']

# 2) Filter to those components and KPIs with anomaly_count > 0
subset = per_kpi[
    (per_kpi['cmdb_id'].isin(top_components)) &
    (per_kpi['anomaly_count'].fillna(0) > 0)
].copy()

# Ensure numeric columns are numeric for sorting
for col in ['anomaly_count', 'total_count', 'max_value', 'p95']:
    subset[col] = pd.to_numeric(subset[col], errors='coerce').fillna(0)

# 2) For each component, take up to top 4 KPI rows sorted by anomaly_count desc, then total_count desc
subset = subset.sort_values(['cmdb_id', 'anomaly_count', 'total_count'], ascending=[True, False, False])
top_per_component = subset.groupby('cmdb_id', sort=False).head(4).reset_index(drop=True)

# 4) Classify resource_type based on kpi_name
def classify_resource(kname):
    if not isinstance(kname, str):
        return 'other'
    s = kname.lower()
    if any(x in s for x in ['memory', 'jvm', 'heap', 'rss', 'swap']):
        return 'memory'
    if any(x in s for x in ['cpu', 'load']):
        return 'cpu'
    if any(x in s for x in ['fs', 'disk', 'space', 'vda']):
        return 'disk'
    if any(x in s for x in ['network', 'packet', 'transmit', 'receive', 'tx', 'rx']):
        return 'network'
    if any(x in s for x in ['process', 'starttime', 'uptime', 'runtime']):
        return 'process'
    return 'other'

top_per_component['resource_type'] = top_per_component['kpi_name'].apply(classify_resource)

# 3) Select required columns in requested order
top_components_kpi_details = top_per_component[
    ['cmdb_id', 'kpi_name', 'total_count', 'anomaly_count', 'earliest_anom_timestamp', 'max_value', 'p95', 'resource_type']
].copy()

# Keep output compact: limit to 20 rows (there will be at most 5*4=20)
top_components_kpi_details = top_components_kpi_details.head(20)

# Display the resulting DataFrame (variable name used to show result)
top_components_kpi_details

```
Out[4]:
```
Summary (plain English):

- I saved the detailed per-KPI rows for the five components in kernel variable top_components_kpi_details.

- Dominant resource issue across these top components: disk-related KPIs (container filesystem limit and usage on /dev/vda1) are the most common anomalies — every one of the five components shows fs_limit anomalies (full count = 31 in-window) and fs_usage anomalies (some had repeated usage breaches). This indicates a widespread disk/space signal on /dev/vda1.

- Other notable resource issues by component (top up to 4 KPIs shown per component):
  - checkoutservice-1
    - disk: container_fs_limit_MB./dev/vda1 (31/31 anomalies)
    - memory: container_memory_max_usage_MB (31/31 anomalies)
    - disk usage: container_fs_usage_MB./dev/vda1 (4 anomalies, earliest 16:27 CST)
    - cpu throttling metric (3 anomalies)
  - checkoutservice-2
    - disk: container_fs_limit_MB./dev/vda1 (31/31)
    - disk usage: container_fs_usage_MB./dev/vda1 (4 anomalies)
    - last_seen / timestamp-like KPI (3 anomalies)
    - a mesh/jaeger-related metric (3 anomalies)
  - frontend-0
    - disk: container_fs_limit_MB./dev/vda1 (31/31)
    - disk usage: container_fs_usage_MB./dev/vda1 (3 anomalies)
    - last_seen (3 anomalies)
    - network transmit MB (3 anomalies)
  - frontend-1
    - disk: container_fs_limit_MB./dev/vda1 (31/31)
    - network transmit MB (5 anomalies, earliest 16:00 CST)
    - cpu usage seconds (4 anomalies)
    - an Istio request-duration metric (4 anomalies)
  - frontend-2
    - disk: container_fs_limit_MB./dev/vda1 (31/31)
    - disk usage: container_fs_usage_MB./dev/vda1 (3 anomalies)
    - last_seen (3 anomalies)
    - network transmit MB (3 anomalies)

- Interpretation: the pervasive disk fs_limit anomalies (and accompanying fs_usage anomalies) point to disk-related issues (space/quota or filesystem reporting) as the dominant signal for these top components. Secondary issues include memory (notably in checkoutservice-1), CPU (some containers show CPU throttling/usage), and network transmit spikes (some frontend nodes).

- Next steps you might take:
  - Inspect /dev/vda1 usage and quotas on the affected nodes/containers and free/extend space.
  - Check per-container disk I/O and filesystem errors, and confirm whether the fs_limit values are expected (they appear equal to large host values).
  - Review memory and CPU throttling for checkoutservice-1 and frontends if service impact is observed.

If you want, I can expand the top_components_kpi_details table or compute counts by resource_type for these five components.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_count  anomaly_count  earliest_anom_timestamp     max_value           p95 resource_type
0   checkoutservice-1   container.node-6.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05          disk
1   checkoutservice-1     container.node-6.container_memory_max_usage_MB           31             31  2022-03-20 16:00:00 CST  1.280039e+02  1.280039e+02        memory
2   checkoutservice-1   container.node-6.container_fs_usage_MB./dev/vda1           31              4  2022-03-20 16:27:00 CST  4.137500e+01  4.136523e+01          disk
3   checkoutservice-1  container.node-6.container_cpu_cfs_throttled_s...           31              3  2022-03-20 16:13:00 CST  4.556703e-01  3.820100e-01           cpu
4   checkoutservice-2   container.node-5.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05          disk
5   checkoutservice-2   container.node-5.container_fs_usage_MB./dev/vda1           31              4  2022-03-20 16:27:00 CST  4.137695e+01  4.136523e+01          disk
6   checkoutservice-2               container.node-5.container_last_seen           31              3  2022-03-20 16:28:00 CST  1.647765e+09  1.647765e+09         other
7   checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...           31              3  2022-03-20 16:00:00 CST  8.050000e-03  2.100000e-03         other
8          frontend-0   container.node-6.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05          disk
9          frontend-0   container.node-6.container_fs_usage_MB./dev/vda1           31              3  2022-03-20 16:23:00 CST  6.992578e+01  6.939844e+01          disk
10         frontend-0               container.node-6.container_last_seen           31              3  2022-03-20 16:28:00 CST  1.647765e+09  1.647765e+09         other
11         frontend-0  container.node-6.container_network_transmit_MB...           31              3  2022-03-20 16:11:00 CST  7.784415e+00  7.690495e+00       network
12         frontend-1   container.node-5.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05          disk
13         frontend-1  container.node-5.container_network_transmit_MB...           31              5  2022-03-20 16:00:00 CST  5.025500e+00  4.620322e+00       network
14         frontend-1       container.node-5.container_cpu_usage_seconds           31              4  2022-03-20 16:21:00 CST  2.133370e-01  1.986080e-01           cpu
15         frontend-1  mesh.source.frontend.adservice.istio_request_d...           31              4  2022-03-20 16:21:00 CST  6.192500e-02  5.585000e-02         other
16         frontend-2   container.node-5.container_fs_limit_MB./dev/vda1           31             31  2022-03-20 16:00:00 CST  6.046307e+05  6.046307e+05          disk
17         frontend-2   container.node-5.container_fs_usage_MB./dev/vda1           31              3  2022-03-20 16:28:00 CST  5.838281e+01  5.785938e+01          disk
18         frontend-2               container.node-5.container_last_seen           31              3  2022-03-20 16:28:00 CST  1.647765e+09  1.647765e+09         other
19         frontend-2  container.node-5.container_network_transmit_MB...           31              3  2022-03-20 16:05:00 CST  9.145580e+00  8.777711e+00       network```
```

In [ ]:
# In[5]:

import re
import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Safely retrieve the per-KPI aggregation DataFrame from kernel
candidate_keys = ['per_kpi', 'result_df', 'agg', 'top_components_kpi_details', 'metric_thresholds']
per_kpi = None
for k in candidate_keys:
    if k in globals():
        df = globals()[k]
        if isinstance(df, pd.DataFrame) and not df.empty:
            per_kpi = df
            break
# If still None, try any DataFrame in globals (fallback)
if per_kpi is None:
    for v in globals().values():
        if isinstance(v, pd.DataFrame) and not v.empty:
            per_kpi = v
            break

if per_kpi is None:
    raise RuntimeError("Per-KPI aggregation DataFrame not found in kernel.")

# Ensure required columns exist; add defaults if missing
for col in ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_timestamp', 'max_value', 'p95', 'resource_type']:
    if col not in per_kpi.columns:
        per_kpi[col] = pd.NA

# Create resource_type if not present or mostly null
if per_kpi['resource_type'].isna().all():
    def classify_resource(kname):
        if not isinstance(kname, str):
            return 'other'
        s = kname.lower()
        if any(x in s for x in ['memory', 'jvm', 'heap', 'rss', 'swap']):
            return 'memory'
        if any(x in s for x in ['cpu', 'load']):
            return 'cpu'
        if any(x in s for x in ['fs', 'disk', 'space', 'vda']):
            return 'disk'
        if any(x in s for x in ['network', 'packet', 'transmit', 'receive', 'tx', 'rx']):
            return 'network'
        if any(x in s for x in ['process', 'starttime', 'uptime', 'runtime']):
            return 'process'
        return 'other'
    per_kpi = per_kpi.copy()
    per_kpi['resource_type'] = per_kpi['kpi_name'].apply(classify_resource)

# Normalize numeric columns
per_kpi['anomaly_count'] = pd.to_numeric(per_kpi['anomaly_count'], errors='coerce').fillna(0).astype(int)
per_kpi['max_value'] = pd.to_numeric(per_kpi['max_value'], errors='coerce')
per_kpi['p95'] = pd.to_numeric(per_kpi['p95'], errors='coerce')

# Helper to parse earliest_anom_timestamp into timezone-aware datetime (Asia/Shanghai)
def parse_ts_to_local(s):
    if pd.isna(s):
        return pd.NaT
    # If it's numeric epoch
    try:
        if isinstance(s, (int, float, np.integer, np.floating)) or (isinstance(s, str) and s.isdigit()):
            return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz)
    except Exception:
        pass
    # Try parsing string
    try:
        dt = pd.to_datetime(str(s), utc=True, errors='coerce')
        if pd.isna(dt):
            dt2 = pd.to_datetime(str(s), errors='coerce')
            if pd.isna(dt2):
                return pd.NaT
            if dt2.tzinfo is None:
                return tz.localize(dt2)
            return dt2.tz_convert(tz)
        return dt.tz_convert(tz)
    except Exception:
        return pd.NaT

per_kpi['_earliest_dt'] = per_kpi['earliest_anom_timestamp'].apply(parse_ts_to_local)

# 1) Filter disk-related rows with kpi_name matching 'usage' or 'fs_usage' or '/dev/vda1'
mask_disk = (
    (per_kpi['resource_type'] == 'disk') &
    per_kpi['kpi_name'].str.contains(r'usage|fs_usage|/dev/vda1', case=False, na=False)
)
disk_usage_anoms = per_kpi.loc[mask_disk, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_timestamp', 'max_value', 'p95', '_earliest_dt']].copy()

# Sort by earliest_anom_timestamp ascending, then anomaly_count desc
disk_usage_anoms = disk_usage_anoms.sort_values(['_earliest_dt', 'anomaly_count'], ascending=[True, False]).reset_index(drop=True)

# Display-limited table (drop helper column)
disk_usage_display = disk_usage_anoms[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_timestamp', 'max_value', 'p95']].head(20)

# 2) Extract hosting node identifier from kpi_name using regex 'node-<number>'
def extract_node(kname):
    if not isinstance(kname, str):
        return pd.NA
    m = re.search(r'node-\d+', kname)
    return m.group(0) if m else pd.NA

disk_usage_anoms['node'] = disk_usage_anoms['kpi_name'].apply(extract_node)

# Per-node summary: node, affected_containers (distinct cmdb_id), earliest_disk_usage_anom (min), total_disk_usage_anoms (sum)
node_group = disk_usage_anoms[~disk_usage_anoms['node'].isna()].copy()
node_disk_summary = (
    node_group
    .groupby('node')
    .agg(
        affected_containers=('cmdb_id', lambda s: s.dropna().unique().size),
        earliest_disk_usage_anom_dt=('_earliest_dt', 'min'),
        total_disk_usage_anoms=('anomaly_count', 'sum')
    )
    .reset_index()
)

# Convert earliest_dt to localized string for display
node_disk_summary['earliest_disk_usage_anom'] = node_disk_summary['earliest_disk_usage_anom_dt'].apply(
    lambda dt: dt.strftime('%Y-%m-%d %H:%M:%S %Z') if not pd.isna(dt) else pd.NA
)
node_disk_summary = node_disk_summary[['node', 'affected_containers', 'earliest_disk_usage_anom', 'total_disk_usage_anoms']]
node_disk_summary = node_disk_summary.sort_values(['affected_containers', 'total_disk_usage_anoms'], ascending=[False, False]).reset_index(drop=True).head(20)

# 3) Totals
num_containers_with_disk_anoms = int(disk_usage_anoms['cmdb_id'].dropna().unique().size)
if disk_usage_anoms['_earliest_dt'].dropna().empty:
    earliest_overall_local_str = pd.NA
else:
    earliest_overall_local_str = disk_usage_anoms['_earliest_dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z')

# Keep variables in kernel: disk_usage_anoms, node_disk_summary
disk_usage_display, node_disk_summary, num_containers_with_disk_anoms, earliest_overall_local_str

```
Out[5]:
```
Summary (plain English):

- I saved the filtered disk-usage anomaly rows in kernel variable disk_usage_anoms and the per-node summary in node_disk_summary.

- What I found (compact):
  - disk_usage_anoms (sample shown) contains many container KPIs like:
    - container.node-6.container_fs_limit_MB./dev/vda1 (anomaly_count = 31 for many containers)
    - container.node-6.container_fs_usage_MB./dev/vda1 (anomaly_count = 31 for some containers)
    - These rows span many containers (examples: adservice-0, adservice-1, adservice-2, checkoutservice-1, frontend-0, etc.).
  - Per-node summary (top results):
    - node-6: affected_containers = 34, earliest_disk_usage_anom = 2022-03-21 00:00:00 CST, total_disk_usage_anoms = 1243
    - node-5: affected_containers = 6, earliest_disk_usage_anom = 2022-03-21 00:00:00 CST, total_disk_usage_anoms = 205

- Totals:
  - Distinct containers with disk usage anomalies: 46
  - Earliest overall disk usage anomaly (across all containers): 2022-03-21 00:00:00 CST

- Interpretation: Disk filesystem metrics for /dev/vda1 (fs_limit and fs_usage) are the dominant anomalies, concentrated primarily on node-6 (34 containers affected) and to a lesser extent node-5. This suggests a node-level disk/space issue impacting many containers.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                          kpi_name  anomaly_count  earliest_anom_timestamp      max_value            p95
0          adservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
1          adservice-0  container.node-6.container_fs_usage_MB./dev/vda1             31  2022-03-20 16:00:00 CST      26.382812      26.382812
2          adservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
3          adservice-2  container.node-5.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
4         adservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
5         adservice2-0  container.node-6.container_fs_usage_MB./dev/vda1             31  2022-03-20 16:00:00 CST      26.382812      26.382812
6        cartservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
7        cartservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
8        cartservice-2  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
9       cartservice2-0  container.node-5.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
10   checkoutservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
11   checkoutservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
12   checkoutservice-2  container.node-5.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
13  checkoutservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
14   currencyservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
15   currencyservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
16   currencyservice-2  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
17  currencyservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
18      emailservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
19      emailservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281,      node  affected_containers earliest_disk_usage_anom  total_disk_usage_anoms
0  node-6                   34  2022-03-21 00:00:00 CST                    1243
1  node-5                    6  2022-03-21 00:00:00 CST                     205, 46, '2022-03-21 00:00:00 CST')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# --- 1) get disk_usage_anoms from kernel and extract node-6 containers ---
if 'disk_usage_anoms' not in globals():
    raise RuntimeError("Required variable 'disk_usage_anoms' not found in kernel.")
disk_df = disk_usage_anoms.copy()

# filter kpi_name containing 'node-6' and usage or /dev/vda1
mask_node6_usage = (
    disk_df['kpi_name'].str.contains('node-6', case=False, na=False) &
    disk_df['kpi_name'].str.contains(r'usage|fs_usage|/dev/vda1', case=False, na=False)
)
node6_containers = sorted(disk_df.loc[mask_node6_usage, 'cmdb_id'].dropna().unique().tolist())

# --- 2) load trace dataframe and filter to incident window ---
# Prefer variable name 'df_trace' from kernel (created earlier). Fall back to 'trace' if needed.
df_trace = globals().get('df_trace', globals().get('trace', None))
if df_trace is None or not isinstance(df_trace, pd.DataFrame):
    raise RuntimeError("Trace dataframe not found in kernel (expected 'df_trace' or 'trace').")

start_ts = 1647763200
end_ts = 1647765000
trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# --- 3) compute global P95 thresholds for traces using full trace series (required BEFORE filtering) ---
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Filter trace_window to node6_containers and merge thresholds
trace_node6 = trace_window[trace_window['cmdb_id'].isin(node6_containers)].copy()
trace_node6 = trace_node6.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Determine anomalies where value > p95 (only when p95 is present)
trace_node6['is_trace_anom'] = False
mask_has_p95 = trace_node6['p95'].notna()
trace_node6.loc[mask_has_p95, 'is_trace_anom'] = trace_node6.loc[mask_has_p95, 'value'] > trace_node6.loc[mask_has_p95, 'p95']

# Aggregation per (cmdb_id, trace_name)
def agg_trace_group(g):
    total = int(g['timestamp'].count())
    anom_count = int(g['is_trace_anom'].sum())
    if anom_count > 0:
        earliest = int(g.loc[g['is_trace_anom'], 'timestamp'].min())
    else:
        earliest = np.nan
    return pd.Series({
        'trace_total_count': total,
        'trace_anom_count': anom_count,
        'earliest_trace_anom_epoch': earliest
    })

node6_trace_metric_agg = trace_node6.groupby(['cmdb_id', 'trace_name']).apply(agg_trace_group).reset_index()

# Convert earliest epoch to localized string
def epoch_to_local_str(e):
    if pd.isna(e):
        return pd.NA
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

node6_trace_metric_agg['earliest_trace_anom'] = node6_trace_metric_agg['earliest_trace_anom_epoch'].apply(epoch_to_local_str)
node6_trace_metric_agg = node6_trace_metric_agg.drop(columns=['earliest_trace_anom_epoch'])

# --- 4) Aggregate per cmdb_id ---
summary = (
    node6_trace_metric_agg
    .groupby('cmdb_id')
    .agg(
        total_trace_anom_points = ('trace_anom_count', 'sum'),
        trace_total_rows = ('trace_total_count', 'sum'),
        number_of_trace_metrics = ('trace_name', 'nunique'),
        earliest_trace_anom_epoch = ('earliest_trace_anom', lambda s: 
            np.nan if s.dropna().empty else int(pd.to_datetime(s.dropna().iloc[0]).tz_localize(None).timestamp()) )
    )
    .reset_index()
)

# The lambda above is simplistic; instead compute earliest properly using available earliest_trace_anom strings
def min_earliest_str(series):
    vals = series.dropna().tolist()
    if not vals:
        return pd.NA
    dts = [pd.to_datetime(v) for v in vals]
    min_dt = min(dts)
    # assume these are timezone-aware strings already; convert to Asia/Shanghai
    if min_dt.tzinfo is None:
        # treat as UTC then convert
        min_dt = min_dt.tz_localize('UTC').tz_convert(tz)
    else:
        min_dt = min_dt.tz_convert(tz)
    return min_dt.strftime('%Y-%m-%d %H:%M:%S %Z')

earliest_per_cmdb = node6_trace_metric_agg.groupby('cmdb_id')['earliest_trace_anom'].agg(min_earliest_str=min_earliest_str).reset_index()
node6_trace_summary = summary.drop(columns=['earliest_trace_anom_epoch']).merge(earliest_per_cmdb, on='cmdb_id', how='left')
node6_trace_summary = node6_trace_summary.rename(columns={'min_earliest_str':'earliest_trace_anom'})

# Sort by total_trace_anom_points desc and limit top 20
node6_trace_summary = node6_trace_summary.sort_values('total_trace_anom_points', ascending=False).reset_index(drop=True).head(20)

# Totals
num_node6_with_trace_anoms = int((node6_trace_summary['total_trace_anom_points'] > 0).sum())
# earliest overall trace anomaly among node-6 containers
all_earliest = node6_trace_metric_agg['earliest_trace_anom'].dropna()
if all_earliest.empty:
    earliest_overall_trace_anom = pd.NA
else:
    earliest_overall_trace_anom = min_earliest_str(all_earliest)

# Save variables in kernel (they are already assigned above)
node6_containers, node6_trace_metric_agg = node6_containers, node6_trace_metric_agg
node6_trace_summary = node6_trace_summary

# Final compact outputs
node6_containers, node6_trace_metric_agg.head(20), node6_trace_summary, num_node6_with_trace_anoms, earliest_overall_trace_anom

```
Out[6]:
```
Summary (plain English):

- I extracted containers on node-6 with filesystem USAGE anomalies and saved them as node6_containers. There are 34 such containers (examples: adservice-0, adservice-1, frontend-0, checkoutservice-1, cartservice-0, etc.).

- I filtered trace data to the incident window (2022-03-20 16:00:00–16:30:00 CST), joined global trace P95 thresholds, and computed per-(cmdb_id, trace_name) anomaly counts (saved as node6_trace_metric_agg). Example rows show per-trace counts and earliest trace anomaly timestamps (e.g., adservice-0 has several traces with 30 rows in-window and a few anomalies with earliest timestamps like 2022-03-20 16:03–16:14 CST).

- I aggregated trace anomalies per container (saved as node6_trace_summary). Top containers by total trace anomaly points (total_trace_anom_points) are:
  1) frontend-0 — 121 trace anomaly points; earliest_trace_anom = 2022-03-21 00:00:00 CST; trace_total_rows = 2756; number_of_trace_metrics = 92
  2) checkoutservice-1 — 94 points; earliest 2022-03-21 00:00:00 CST; trace_total_rows = 2640; metrics = 92
  3) checkoutservice-0 — 91 points; earliest 2022-03-21 00:00:00 CST; trace_total_rows = 2648; metrics = 92
  4) productcatalogservice-0 — 51 points; earliest 2022-03-21 00:00:00 CST
  5) productcatalogservice-2 — 50 points; earliest 2022-03-21 00:01:00 CST
  (list continues — top 20 saved in node6_trace_summary).

- Totals and key facts:
  - Number of node-6 containers with any trace anomaly (total_trace_anom_points > 0): 20.
  - Earliest overall trace anomaly among these node-6 containers: 2022-03-21 00:00:00 CST.

- Saved kernel variables for further inspection:
  - node6_containers (list of 34 containers)
  - node6_trace_metric_agg (per-(cmdb_id, trace_name) metrics with trace_total_count, trace_anom_count, earliest_trace_anom)
  - node6_trace_summary (per-container aggregated trace anomaly summary)

If you want, I can show the full node6_trace_summary table (up to 20 rows) or aggregate trace anomaly counts by resource or trace type.

The original code execution output of IPython Kernel is also provided below for reference:

(['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice2-0'],         cmdb_id                           trace_name  trace_total_count  trace_anom_count      earliest_trace_anom
0   adservice-0  trace.from_frontend-0.duration_mean               30.0               2.0  2022-03-20 16:11:00 CST
1   adservice-0   trace.from_frontend-0.duration_p95               30.0               2.0  2022-03-20 16:14:00 CST
2   adservice-0     trace.from_frontend-0.error_rate               30.0               0.0                     <NA>
3   adservice-0      trace.from_frontend-0.row_count               30.0               2.0  2022-03-20 16:12:00 CST
4   adservice-0  trace.from_frontend-1.duration_mean               30.0               2.0  2022-03-20 16:03:00 CST
5   adservice-0   trace.from_frontend-1.duration_p95               30.0               2.0  2022-03-20 16:17:00 CST
6   adservice-0     trace.from_frontend-1.error_rate               30.0               0.0                     <NA>
7   adservice-0      trace.from_frontend-1.row_count               30.0               1.0  2022-03-20 16:29:00 CST
8   adservice-0  trace.from_frontend-2.duration_mean               30.0               2.0  2022-03-20 16:14:00 CST
9   adservice-0   trace.from_frontend-2.duration_p95               30.0               2.0  2022-03-20 16:18:00 CST
10  adservice-0     trace.from_frontend-2.error_rate               30.0               0.0                     <NA>
11  adservice-0      trace.from_frontend-2.row_count               30.0               2.0  2022-03-20 16:05:00 CST
12  adservice-1  trace.from_frontend-0.duration_mean               30.0               2.0  2022-03-20 16:03:00 CST
13  adservice-1   trace.from_frontend-0.duration_p95               30.0               2.0  2022-03-20 16:03:00 CST
14  adservice-1     trace.from_frontend-0.error_rate               30.0               0.0                     <NA>
15  adservice-1      trace.from_frontend-0.row_count               30.0               2.0  2022-03-20 16:12:00 CST
16  adservice-1  trace.from_frontend-1.duration_mean               30.0               2.0  2022-03-20 16:02:00 CST
17  adservice-1   trace.from_frontend-1.duration_p95               30.0               2.0  2022-03-20 16:02:00 CST
18  adservice-1     trace.from_frontend-1.error_rate               30.0               0.0                     <NA>
19  adservice-1      trace.from_frontend-1.row_count               30.0               2.0  2022-03-20 16:25:00 CST,                      cmdb_id  total_trace_anom_points  trace_total_rows  number_of_trace_metrics      earliest_trace_anom
0                 frontend-0                    121.0            2756.0                       92  2022-03-21 00:00:00 CST
1          checkoutservice-1                     94.0            2640.0                       92  2022-03-21 00:00:00 CST
2          checkoutservice-0                     91.0            2648.0                       92  2022-03-21 00:00:00 CST
3    productcatalogservice-0                     51.0            1200.0                       40  2022-03-21 00:00:00 CST
4    productcatalogservice-2                     50.0            1200.0                       40  2022-03-21 00:01:00 CST
5                frontend2-0                     49.0            1076.0                       36  2022-03-21 00:01:00 CST
6    productcatalogservice-1                     49.0            1200.0                       40  2022-03-21 00:00:00 CST
7         checkoutservice2-0                     41.0            1044.0                       36  2022-03-21 00:01:00 CST
8    recommendationservice-1                     38.0             840.0                       28  2022-03-21 00:03:00 CST
9    recommendationservice-2                     37.0             840.0                       28  2022-03-21 00:01:00 CST
10   recommendationservice-0                     36.0             840.0                       28  2022-03-21 00:00:00 CST
11         currencyservice-1                     30.0             720.0                       24  2022-03-21 00:02:00 CST
12         currencyservice-0                     28.0             720.0                       24  2022-03-21 00:01:00 CST
13  recommendationservice2-0                     26.0             600.0                       20  2022-03-21 00:02:00 CST
14         currencyservice-2                     26.0             720.0                       24  2022-03-21 00:03:00 CST
15         shippingservice-1                     26.0             676.0                       24  2022-03-21 00:01:00 CST
16         shippingservice-0                     24.0             680.0                       24  2022-03-21 00:00:00 CST
17             cartservice-1                     20.0             840.0                       28  2022-03-21 00:01:00 CST
18             cartservice-0                     19.0             840.0                       28  2022-03-21 00:01:00 CST
19             cartservice-2                     18.0             840.0                       28  2022-03-21 00:01:00 CST, 20, '2022-03-21 00:00:00 CST')```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse globals: df_log, node6_containers, df_error_logs (from prior steps)
if 'df_log' not in globals():
    raise RuntimeError("df_log not found in kernel.")
if 'node6_containers' not in globals():
    raise RuntimeError("node6_containers not found in kernel.")
df_log = globals()['df_log']
node6_containers = globals()['node6_containers']
df_error_logs = globals().get('df_error_logs', pd.DataFrame(columns=['timestamp','cmdb_id','message']))

# Incident window
start_ts = 1647763200
end_ts = 1647765000

# 1) Filter logs to window and node6_containers, save as node6_logs_window
node6_logs_window = df_log[
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts) &
    (df_log['cmdb_id'].isin(node6_containers))
].copy()

# Normalize numeric value column
if 'value' in node6_logs_window.columns:
    node6_logs_window['value'] = pd.to_numeric(node6_logs_window['value'], errors='coerce')
else:
    node6_logs_window['value'] = np.nan

# 2) Per (cmdb_id, log_name) aggregations
grp = node6_logs_window.groupby(['cmdb_id', 'log_name'])

agg_counts = grp['timestamp'].agg(total_rows='count').reset_index()
agg_max = grp['value'].agg(max_value='max').reset_index()

# earliest timestamp where value > 0
pos = node6_logs_window[node6_logs_window['value'] > 0]
earliest_pos = pos.groupby(['cmdb_id', 'log_name'])['timestamp'].min().reset_index(name='earliest_pos_ts')

# Merge aggregates
log_agg = agg_counts.merge(agg_max, on=['cmdb_id','log_name'], how='left')
log_agg = log_agg.merge(earliest_pos, on=['cmdb_id','log_name'], how='left')

# Convert earliest_pos_ts epoch to Asia/Shanghai localized string, keep NA as pd.NA
def epoch_to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return pd.NA

log_agg['earliest_timestamp_where_value_gt0'] = log_agg['earliest_pos_ts'].apply(epoch_to_local_str)
log_agg = log_agg.drop(columns=['earliest_pos_ts'])

# Sort by total_rows desc and limit 20 rows for display
log_agg_sorted = log_agg.sort_values(['total_rows', 'max_value'], ascending=[False, False]).reset_index(drop=True)
log_agg_display = log_agg_sorted[['cmdb_id','log_name','total_rows','max_value','earliest_timestamp_where_value_gt0']].head(20)

# 3) Total number of node-6 containers that have any log rows in the window
containers_with_logs_count = int(node6_logs_window['cmdb_id'].dropna().unique().size)

# 4) Count error_logs rows for those containers within the same window
if not df_error_logs.empty:
    err_count = df_error_logs[
        (df_error_logs['cmdb_id'].isin(node6_containers)) &
        (df_error_logs['timestamp'] >= start_ts) &
        (df_error_logs['timestamp'] <= end_ts)
    ].shape[0]
else:
    err_count = 0

# Save node6_logs_window in kernel (already assigned)
node6_logs_window = node6_logs_window

# Final compact outputs (variables displayed, not printed)
log_agg_display, node6_logs_window.shape, containers_with_logs_count, err_count

```
Out[7]:
```
Summary (plain English):

- I filtered logs to the incident window and to node-6 containers; the filtered log subset is saved as node6_logs_window (shape: 1524 rows, 4 columns).

- Top log metrics (compact sample, sorted by total_rows):
  - frontend-0 — log.row_count: total_rows=31, max_value=2657, earliest value>0 at 2022-03-20 16:00:00 CST
  - productcatalogservice-1 — log.row_count: total_rows=31, max_value=1420, earliest 2022-03-20 16:00:00 CST
  - productcatalogservice-0 — log.row_count: total_rows=31, max_value=1285, earliest 2022-03-20 16:00:00 CST
  - cartservice-1 — log.row_count: total_rows=31, max_value=1139, earliest 2022-03-20 16:00:00 CST
  - cartservice-0 — log.row_count: total_rows=31, max_value=1132, earliest 2022-03-20 16:00:00 CST

- Observations:
  - Many node-6 containers show log.row_count present for every minute in the window (31 rows), with nonzero max counts at the start of the window.
  - log.error_count is present for many containers but its max_value is 0 in this window (no positive error counts observed in the sample rows).

- Counts and state:
  - Number of node-6 containers that have any log rows in the window: 25.
  - Number of error_logs.csv rows for those containers in the same window: 0 (no error_log entries).

If you want, I can show the full node6_logs_window (up to 200 rows) or aggregate log metrics by resource/component.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  total_rows  max_value earliest_timestamp_where_value_gt0
0                frontend-0    log.row_count          31     2657.0            2022-03-20 16:00:00 CST
1   productcatalogservice-1    log.row_count          31     1420.0            2022-03-20 16:00:00 CST
2   productcatalogservice-0    log.row_count          31     1285.0            2022-03-20 16:00:00 CST
3             cartservice-1    log.row_count          31     1139.0            2022-03-20 16:00:00 CST
4             cartservice-0    log.row_count          31     1132.0            2022-03-20 16:00:00 CST
5             cartservice-2    log.row_count          31     1109.0            2022-03-20 16:00:00 CST
6         currencyservice-1    log.row_count          31     1020.0            2022-03-20 16:00:00 CST
7         currencyservice-2    log.row_count          31     1013.0            2022-03-20 16:00:00 CST
8         currencyservice-0    log.row_count          31     1007.0            2022-03-20 16:00:00 CST
9   recommendationservice-2    log.row_count          31      273.0            2022-03-20 16:00:00 CST
10  recommendationservice-0    log.row_count          31      272.0            2022-03-20 16:00:00 CST
11        shippingservice-0    log.row_count          31      131.0            2022-03-20 16:00:00 CST
12            cartservice-0  log.error_count          31        0.0                               <NA>
13            cartservice-1  log.error_count          31        0.0                               <NA>
14            cartservice-2  log.error_count          31        0.0                               <NA>
15        currencyservice-0  log.error_count          31        0.0                               <NA>
16        currencyservice-1  log.error_count          31        0.0                               <NA>
17        currencyservice-2  log.error_count          31        0.0                               <NA>
18               frontend-0  log.error_count          31        0.0                               <NA>
19  productcatalogservice-0  log.error_count          31        0.0                               <NA>, (1524, 4), 25, 0)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse disk_usage_anoms from kernel
if 'disk_usage_anoms' not in globals():
    raise RuntimeError("disk_usage_anoms not found in kernel.")
df = disk_usage_anoms.copy()

# 1) Filter rows: kpi_name contains 'usage' or '/dev/vda1' and contains 'node-6'
mask = df['kpi_name'].str.contains(r'usage|/dev/vda1', case=False, na=False) & df['kpi_name'].str.contains(r'node-6', case=False, na=False)
node6_fs_usage_rows = df.loc[mask].reset_index(drop=True)

# Helper: robust parse of stored earliest_anom_timestamp strings to Asia/Shanghai tz-aware datetimes
def parse_to_asia_shanghai(s):
    if pd.isna(s):
        return pd.NaT
    try:
        s_str = str(s).strip()
        # Remove trailing timezone tokens like 'CST' to avoid ambiguous parsing, then localize as Asia/Shanghai
        s_clean = re.sub(r'\s+[A-Za-z]+$', '', s_str)
        dt = pd.to_datetime(s_clean, errors='coerce')
        if pd.isna(dt):
            return pd.NaT
        # if dt has no tzinfo, localize to Asia/Shanghai
        if dt.tzinfo is None:
            return tz.localize(dt)
        # else convert to Asia/Shanghai
        return dt.tz_convert(tz)
    except Exception:
        return pd.NaT

# Parse earliest timestamps to datetimes for comparisons
node6_fs_usage_rows['_earliest_dt'] = node6_fs_usage_rows['earliest_anom_timestamp'].apply(parse_to_asia_shanghai)

# 2) Earliest overall earliest_anom_timestamp (minimum)
if node6_fs_usage_rows['_earliest_dt'].dropna().empty:
    earliest_overall_epoch = pd.NA
    earliest_overall_local = pd.NA
else:
    earliest_dt = node6_fs_usage_rows['_earliest_dt'].min()
    earliest_overall_epoch = int(earliest_dt.timestamp())
    earliest_overall_local = earliest_dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 3) Number of distinct cmdb_id on node-6 that have anomaly_count>0 for these usage KPIs, and total sum of their anomaly_count
# Consider a container "has anomaly" if any of its filtered KPI rows have anomaly_count>0
per_container = (
    node6_fs_usage_rows
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_anom_points=('anomaly_count', 'sum'),
        any_anom=('anomaly_count', lambda s: int((s>0).any())),
        earliest_dt_container=('_earliest_dt', 'min')
    )
)
containers_with_anom = per_container[per_container['any_anom'] > 0].copy()
num_containers_with_anom = int(containers_with_anom.shape[0])
total_anom_points = int(containers_with_anom['total_anom_points'].sum())

# 4) Up to 20 cmdb_id values (sorted) that have earliest_anom_timestamp equal to the earliest overall timestamp
if pd.isna(earliest_overall_local):
    containers_at_earliest = pd.DataFrame(columns=['cmdb_id','total_anom_points','earliest_anom_timestamp'])
else:
    # Compare per-container earliest (tz-aware) to earliest_dt
    containers_at_earliest = per_container[per_container['earliest_dt_container'] == earliest_dt].copy()
    containers_at_earliest['earliest_anom_timestamp'] = containers_at_earliest['earliest_dt_container'].apply(
        lambda d: d.strftime('%Y-%m-%d %H:%M:%S %Z') if not pd.isna(d) else pd.NA
    )
    containers_at_earliest = containers_at_earliest[['cmdb_id','total_anom_points','earliest_anom_timestamp']].sort_values('cmdb_id').head(20).reset_index(drop=True)

# Prepare compact displays
node6_fs_usage_rows_display = node6_fs_usage_rows[['cmdb_id','kpi_name','anomaly_count','earliest_anom_timestamp','max_value','p95']].head(20)

# Export variables (saved in kernel)
node6_fs_usage_rows = node6_fs_usage_rows

# Final compact outputs
node6_fs_usage_rows_display, earliest_overall_epoch, earliest_overall_local, num_containers_with_anom, total_anom_points, containers_at_earliest

```
Out[8]:
```
Summary (plain English):

- Filtered node-6 filesystem USAGE rows saved as variable: node6_fs_usage_rows.

- Earliest overall disk-usage anomaly:
  - Epoch: 1647763200
  - Local time (Asia/Shanghai): 2022-03-20 16:00:00 CST

- Scope:
  - Distinct node-6 containers with usage anomalies: 34
  - Total sum of anomaly_count across those containers: 1,243

- Up to 20 containers whose per-container earliest anomaly equals that earliest timestamp (cmdb_id — total_anom_points — earliest_anom_timestamp):
  1) adservice-0 — 62 — 2022-03-20 16:00:00 CST
  2) adservice-1 — 31 — 2022-03-20 16:00:00 CST
  3) adservice2-0 — 62 — 2022-03-20 16:00:00 CST
  4) cartservice-0 — 34 — 2022-03-20 16:00:00 CST
  5) cartservice-1 — 34 — 2022-03-20 16:00:00 CST
  6) cartservice-2 — 35 — 2022-03-20 16:00:00 CST
  7) checkoutservice-0 — 35 — 2022-03-20 16:00:00 CST
  8) checkoutservice-1 — 35 — 2022-03-20 16:00:00 CST
  9) checkoutservice2-0 — 36 — 2022-03-20 16:00:00 CST
 10) currencyservice-0 — 34 — 2022-03-20 16:00:00 CST
 11) currencyservice-1 — 35 — 2022-03-20 16:00:00 CST
 12) currencyservice-2 — 31 — 2022-03-20 16:00:00 CST
 13) currencyservice2-0 — 34 — 2022-03-20 16:00:00 CST
 14) emailservice-0 — 37 — 2022-03-20 16:00:00 CST
 15) emailservice-1 — 35 — 2022-03-20 16:00:00 CST
 16) emailservice-2 — 34 — 2022-03-20 16:00:00 CST
 17) emailservice2-0 — 36 — 2022-03-20 16:00:00 CST
 18) frontend-0 — 34 — 2022-03-20 16:00:00 CST
 19) frontend2-0 — 35 — 2022-03-20 16:00:00 CST
 20) paymentservice-0 — 34 — 2022-03-20 16:00:00 CST

- Interpretation: a node-level filesystem signal on /dev/vda1 started at the very start of the incident window (16:00:00 CST) and affected many containers on node-6 (most containers show both fs_limit and fs_usage anomalies, often 31 points each).

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                          kpi_name  anomaly_count  earliest_anom_timestamp      max_value            p95
0          adservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
1          adservice-0  container.node-6.container_fs_usage_MB./dev/vda1             31  2022-03-20 16:00:00 CST      26.382812      26.382812
2          adservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
3         adservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
4         adservice2-0  container.node-6.container_fs_usage_MB./dev/vda1             31  2022-03-20 16:00:00 CST      26.382812      26.382812
5        cartservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
6        cartservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
7        cartservice-2  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
8    checkoutservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
9    checkoutservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
10  checkoutservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
11   currencyservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
12   currencyservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
13   currencyservice-2  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
14  currencyservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
15      emailservice-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
16      emailservice-1  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
17      emailservice-2  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
18     emailservice2-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281
19          frontend-0  container.node-6.container_fs_limit_MB./dev/vda1             31  2022-03-20 16:00:00 CST  604630.738281  604630.738281, 1647763200, '2022-03-20 16:00:00 CST', 34, 1243,                cmdb_id  total_anom_points  earliest_anom_timestamp
0          adservice-0                 62  2022-03-20 16:00:00 CST
1          adservice-1                 31  2022-03-20 16:00:00 CST
2         adservice2-0                 62  2022-03-20 16:00:00 CST
3        cartservice-0                 34  2022-03-20 16:00:00 CST
4        cartservice-1                 34  2022-03-20 16:00:00 CST
5        cartservice-2                 35  2022-03-20 16:00:00 CST
6    checkoutservice-0                 35  2022-03-20 16:00:00 CST
7    checkoutservice-1                 35  2022-03-20 16:00:00 CST
8   checkoutservice2-0                 36  2022-03-20 16:00:00 CST
9    currencyservice-0                 34  2022-03-20 16:00:00 CST
10   currencyservice-1                 35  2022-03-20 16:00:00 CST
11   currencyservice-2                 31  2022-03-20 16:00:00 CST
12  currencyservice2-0                 34  2022-03-20 16:00:00 CST
13      emailservice-0                 37  2022-03-20 16:00:00 CST
14      emailservice-1                 35  2022-03-20 16:00:00 CST
15      emailservice-2                 34  2022-03-20 16:00:00 CST
16     emailservice2-0                 36  2022-03-20 16:00:00 CST
17          frontend-0                 34  2022-03-20 16:00:00 CST
18         frontend2-0                 35  2022-03-20 16:00:00 CST
19    paymentservice-0                 34  2022-03-20 16:00:00 CST)```
```